# Champions League + MLflow en Databricks
Andy Fuentes · Diederich Solís · Diego Linares · Christian Echeverria

Adaptación del ZIP y del [Taller 3 original](https://github.com/DiegoLinares11/MLFlow). [Repositorio de esta entrega](https://github.com/Andyfer004/Ejercicio5-MLOPS). Las métricas del notebook 03 son históricas del taller; este notebook se entrega sin resultados de Databricks.

## 1. Entorno
Instale este proyecto según [README](../README.md). En Databricks use un Git folder de **Andyfer004/Ejercicio5-MLOPS** y un compute Python. En una celda separada ejecute `%pip install -e "/Workspace/Users/SU_USUARIO/Ejercicio5-MLOPS"` con la ruta real; reinicie Python si se solicita. En Jupyter local use el kernel del entorno virtual instalado. El dataset se lee del paquete, independientemente del directorio de trabajo.

In [ ]:
from pathlib import Path
import sys

# Localiza el script desde la raíz o desde notebooks/.
RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pipeline_champions_databricks.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra el notebook desde el Git folder de este proyecto.")
sys.path.insert(0, str(RAIZ))
from pipeline_champions_databricks import ejecutar
from taller3_pipeline.modelos import FAMILIAS
print("Familias disponibles:", list(FAMILIAS))

## 2. Comprensión y preparación de datos (CRISP-DM)
Se eliminan filas vacías, score y winner; el pipeline transforma porcentajes, razones y equipos dentro de cada fold. Las estadísticas de juego son posteriores al encuentro: este ejercicio clasifica resultados con estadísticas observadas y no demuestra capacidad de pronóstico previo.

In [ ]:
from taller3_pipeline.data import extraer_datos, filtrar_datos, cargar_datos
crudos = extraer_datos()
limpios = filtrar_datos(crudos)
X_train, X_test, y_train, y_test = cargar_datos()
print({"filas_crudas": len(crudos), "partidos": len(limpios),
       "entrenamiento": len(X_train), "prueba": len(X_test)})
print(y_train.value_counts())

## 3. Configuración
En Databricks configure `URI = "databricks"` y una ruta real `/Users/su_correo/Champions_Resultado_Databricks`. En local deje la configuración siguiente. La primera prueba usa 2 iteraciones y 3 folds; para la búsqueda completa cambie a 40 y 5. No configura ni registra un modelo en Unity Catalog por defecto.

In [ ]:
URI = "sqlite:///" + str(RAIZ / "mlflow.db")
EXPERIMENTO = "Champions_Local_Notebook"
FAMILIA = "svc"
N_ITER = 2
FOLDS = 3
MODELO_UC = None  # Opcional en Databricks: catalogo.esquema.modelo

## 4. Modelado, evaluación y MLflow Models
Experiments organiza los runs; Tracking guarda parámetros, métricas, datasets y artefactos. Models empaqueta el pipeline completo con firma y código. Se recarga el modelo y se comprueba su F1 de prueba. La selección de hiperparámetros usa F1 macro de validación cruzada, sin ajustar sobre el conjunto de prueba.

In [ ]:
resultado = ejecutar(EXPERIMENTO, familia=FAMILIA, n_iter=N_ITER, folds=FOLDS,
                     modelo_uc=MODELO_UC, uri=URI, formato="notebook")
assert resultado["recarga_verificada"]
resultado

## 5. Evidencia y entrega
Revise en Experiments el run: `cv_f1_macro`, `test_accuracy`, `test_f1_macro`, baseline, datasets y modelo. Abra `evaluacion/verificacion_recarga.json`, la matriz de confusión y `busqueda/cv_results.csv`. Models no implica que exista un endpoint ni una versión en Unity Catalog. Ejecute también el script según [GUIA_EJECUCION](../GUIA_EJECUCION.md). Después de ejecutar en Databricks, comparta el notebook con el docente, copie su URL real y agréguela al README y al reporte. No se ha ejecutado en Databricks durante esta preparación.